# 01 · Índice de consumo energético: evolución nacional

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santiagoriverti/consumo_energetico_argentina/blob/main/notebooks/01_evolucion_nacional.ipynb)

Mide cuánta energía consume la **industria** (y los **hogares**) de Argentina mes a mes, sumando
electricidad y gas natural en una misma unidad física (terajoules, TJ).

- **Electricidad**: CAMMESA, demanda por agente y tarifa (2012 →). Industria = grandes usuarios del mercado
  mayorista (GUMA, GUME, GUPA, autogeneradores) + usuarios de las distribuidoras con demanda ≥ 300 kW.
- **Gas**: ENARGAS, gas entregado por tipo de usuario (1993 →). Industria = usuarios industriales de las
  distribuidoras + *by-pass* (industrias que toman el gas directo de los gasoductos).
- Índice base **2016 = 100** (la misma base que el IPI manufacturero del INDEC). Desestacionalización STL.

Detalle de fuentes, decisiones y limitaciones: `README.md` y `CONTEXTO.md` del repo.

In [ ]:
# Configuracion: en Colab clona (o actualiza) el repo; en local usa la carpeta del repo
import os, sys, subprocess, importlib
REPO = 'https://github.com/santiagoriverti/consumo_energetico_argentina.git'
if 'google.colab' in sys.modules:
    RAIZ = '/content/consumo_energetico_argentina'
    if os.path.exists(RAIZ):
        # Trae la ultima version de GitHub y descarta cambios locales (la copia de Colab es descartable)
        subprocess.run(['git', '-C', RAIZ, 'fetch', '-q', '--depth', '1', 'origin', 'main'], check=True)
        subprocess.run(['git', '-C', RAIZ, 'reset', '-q', '--hard', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, RAIZ], check=True)
else:  # subir desde la carpeta actual hasta encontrar el repo
    RAIZ = os.getcwd()
    while not os.path.exists(os.path.join(RAIZ, 'src', 'pipeline.py')) and os.path.dirname(RAIZ) != RAIZ:
        RAIZ = os.path.dirname(RAIZ)
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
# Si el notebook ya se corrio en esta sesion, olvidar los modulos de src cargados (pueden ser viejos)
for m in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]
importlib.invalidate_caches()
v = subprocess.run(['git', '-C', RAIZ, 'log', '-1', '--format=%h (%ad) %s', '--date=short'],
                   capture_output=True, text=True).stdout.strip()
print('Version del repo:', v or 'sin git')

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import pipeline, graficos, indices
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
pd.set_option('display.max_columns', 30)

In [ ]:
# ACTUALIZAR = True vuelve a bajar CAMMESA y ENARGAS (~100 MB, unos minutos). Con False usa las tablas
# del repo (data/processed/), que se actualizan con scripts/construir.py.
ACTUALIZAR = False
R = pipeline.calcular(actualizar=ACTUALIZAR)
print(R['meta'])

## El índice industrial

La serie original tiene mucha estacionalidad: en invierno el gas para la industria cae porque las
distribuidoras priorizan a los hogares (contratos interrumpibles). La tendencia-ciclo resume la dirección.

In [ ]:
fig = graficos.g01_indice_industrial(R); plt.show()

In [ ]:
i, v = R['indices'], R['variaciones']
cols = ['industria', 'industria_desest', 'industria_tendencia', 'industria_electricidad', 'industria_gas']
t = i[cols].join(v[['industria', 'industria_contrib_electricidad', 'industria_contrib_gas']], rsuffix='_var_ia_%')
display(t.dropna(subset=['industria']).tail(13))

**Ojo con oct-2019 a mar-2020.** Un único gran usuario clasificado como "Otras industrias" en el área de
Camuzzi Gas Pampeana (provincia de Buenos Aires) multiplica por 20 su consumo de gas (hasta ~280 millones de
m³ por mes) y explica casi todo el pico del índice antes de la pandemia. No parece consumo manufacturero:
está documentado como anomalía a resolver en `CONTEXTO.md`.

## Electricidad y gas

En energía, el gas es ~3/4 del consumo industrial. Las dos fuentes no se mueven igual: la electricidad
sigue de cerca la actividad; el gas además refleja cortes de invierno, cambios de combustible y unos pocos
usuarios muy grandes (refinerías, siderurgia, petroquímica).

In [ ]:
fig = graficos.g02_componentes_industria(R); plt.show()

In [ ]:
fig = graficos.g05_contribuciones(R); plt.show()

In [ ]:
a = R['anual']
display((a[['industria_electricidad', 'industria_gas', 'industria_total', 'industria_part_elec_%']]
         .dropna().assign(var_total_pct=lambda x: 100 * x['industria_total'].pct_change())))

## Energía vs producción industrial

Mismo año base que el IPI manufacturero del INDEC: si la energía cae más que la producción, la industria se
vuelve menos intensiva en energía (o cambió la composición hacia ramas menos intensivas).

In [ ]:
fig = graficos.g03_industria_vs_ipi(R); plt.show()

In [ ]:
x = i[['industria_desest', 'ipi_manufacturero_desest', 'intensidad_industria']].dropna()
display(x.groupby(x.index.year).mean())

## Ramas industriales (gas de grandes usuarios)

ENARGAS informa el gas de los grandes usuarios industriales de las distribuidoras por rama de actividad (no
incluye el *by-pass*). Es el único corte por rama disponible con datos abiertos mensuales.

In [ ]:
fig = graficos.g09_ramas(R); plt.show()

In [ ]:
display(R['ramas']['resumen'])

## Hogares

Consumo residencial de electricidad y gas. Depende sobre todo del clima (calefacción y aire acondicionado) y
de las tarifas; el gas por usuario descuenta el crecimiento de la cantidad de usuarios conectados.

In [ ]:
fig = graficos.g04_hogares(R); plt.show()

In [ ]:
fig = graficos.g10_gas_por_usuario(R); plt.show()

In [ ]:
display(R['anual'][['hogares_electricidad', 'hogares_gas', 'hogares_total', 'hogares_part_elec_%']].dropna())

## Descargar todos los resultados

ZIP con el Excel (una hoja por tabla), los gráficos en PNG (`graficos/`) y las tablas en CSV
(`datos/`). En Colab se descarga solo; en la PC queda en la carpeta `_descargas/` del repo.

In [ ]:
from src.exportar import zip_resultados
zip_path = zip_resultados(R, 'ICE_01_evolucion_nacional')
print('ZIP:', zip_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(zip_path)